In [ ]:
#| default_exp trace

In [ ]:
#| hide
from fastcore.test import *

# Run trace

A traced run shows what each line of a cell did, for one real run. `trace_cell` runs the code under `sys.settrace`, looks only at frames of the cell itself, and records the names each line changed. Kunda ships this function into the kernel as text, so it uses nothing but the standard library (and `dhrishti`, when the kernel has it, to clip the repr of unknown objects).

A run is cheap to bound and costs nothing when you do not ask for it: no hook is set until `run_traced` is called, and the hook is removed in a `finally`.

In [ ]:
#| export
import ast, asyncio, inspect, json
from fastcore.all import AttrDict, patch
from kunda.kernel import MessageRouter

## The tracer

`trace_cell(code, ns)` compiles `code`, runs it in the namespace `ns`, and returns a dict. It keeps these limits:

- `keep`: of one line, the first `keep` and the last `keep` runs are kept. The runs in between are a counter (an event named `skip`).
- `max_seen`: stop tracing after this many events, and `secs`: stop after this many seconds. The cell goes on running without the hook.
- `max_events`: the most events in the answer.
- `max_repr`: the longest value text.

Each event is `{step, line, event, depth, locals_changed}`. `event` is `line`, `call`, `return` or `exception`. For a `line` event, `locals_changed` holds what the names became *after* that line ran, so one row of the table is one line and its effect.

In [ ]:
#| export
def trace_cell(code, ns, watch=None, max_events=2000, max_repr=200, secs=5., keep=50, max_seen=200_000, fn=None):
    "Run `code` in `ns` under a bounded line tracer and return `{events, error, seen, truncated, reason, source}`."
    import ast, sys, time, reprlib, types, collections, linecache, uuid
    fn = fn or f'<kunda-trace-{uuid.uuid4().hex[:6]}>'
    try:
        from IPython import get_ipython
        ip = get_ipython()
        if ip is not None: code = ip.input_transformer_manager.transform_cell(code)
    except Exception: pass
    tree = ast.parse(code, fn)
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
    stored = {n.id for n in ast.walk(tree) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}
    stored |= {a.asname or a.name.split('.')[0] for n in ast.walk(tree) if isinstance(n, (ast.Import, ast.ImportFrom)) for a in n.names}
    if last is not None: stored |= {n.id for n in ast.walk(last) if isinstance(n, ast.NamedExpr) for n in [n.target]}
    body = compile(tree, fn, 'exec')
    tail = compile(ast.Expression(last.value), fn, 'eval') if last is not None else None
    linecache.cache[fn] = (len(code), None, code.splitlines(True), fn)
    want = set(watch) if watch else None
    R = reprlib.Repr()
    R.maxlist = R.maxtuple = R.maxset = R.maxfrozenset = R.maxdeque = R.maxarray = 20
    R.maxdict, R.maxlevel, R.maxstring, R.maxother, R.maxlong = 10, 3, max_repr, max_repr, max_repr
    R.repr_instance = lambda x, level: f'<{type(x).__name__}>'
    PLAIN = (int, float, bool, complex, type(None), str, bytes, list, tuple, dict, set, frozenset, collections.deque, range)
    SKIP = (types.FunctionType, types.ModuleType, types.BuiltinFunctionType, types.MethodType, type)
    def clip(s): return s if len(s) <= max_repr else s[:max_repr - 1] + '\u2026'
    def rep(v):
        try:
            if type(v) in PLAIN: return clip(R.repr(v))
            try:
                from dhrishti.core import safe_repr
                return clip(safe_repr(v, max_repr))
            except ImportError: return f'<{type(v).__name__}>'
        except Exception as e: return f'<{type(v).__name__}: repr failed>'
    def names(frame):
        loc = frame.f_locals
        if frame.f_code.co_name == '<module>': ks = [k for k in (want or stored) if k in loc]
        else: ks = [k for k in loc if want is None or k in want]
        return [(k, loc[k]) for k in ks if not k.startswith(('__', '.')) and not isinstance(loc[k], SKIP)]
    def snap(frame, old):
        "Return the names whose value differs from `old` (updated in place), as `{name: text}`."
        out = {}
        for k, v in names(frame):
            key = rep(v) if type(v) in PLAIN else ('id', id(v))
            o = old.get(k)
            if o is not None and o[0] == key: continue
            txt = key if type(key) is str else rep(v)
            old[k] = (key, txt)
            out[k] = txt
        return out
    st = types.SimpleNamespace(step=0, seen=0, stop=False, reason=None, depth=0, t0=time.perf_counter())
    events, cnt, ring, skips, frames, modsnap = [], {}, {}, {}, {}, {}
    def add(rec):
        k = (rec['event'], rec['line'])
        c = cnt[k] = cnt.get(k, 0) + 1
        if c <= keep: events.append(rec)
        else:
            if c == keep + 1:
                sk = skips[k] = dict(step=rec['step'], line=rec['line'], event='skip', of=rec['event'], count=0, depth=rec['depth'], locals_changed={})
                events.append(sk)
            q = ring.setdefault(k, collections.deque(maxlen=keep))
            if len(q) == keep: skips[k]['locals_changed'].update(q[0]['locals_changed'])
            q.append(rec)
    def over():
        st.seen += 1
        if st.seen > max_seen: st.reason = f'stopped after {max_seen} events'
        elif not st.seen & 63 and time.perf_counter() - st.t0 > secs: st.reason = f'stopped after {secs} seconds'
        if st.reason: st.stop = True
        return st.stop
    def rec(event, frame, **ch):
        st.step += 1
        r = dict(step=st.step, line=frame.f_lineno, event=event, depth=st.depth, locals_changed=ch)
        add(r)
        return r
    def local(frame, event, arg):
        if st.stop or over():
            sys.settrace(None); frame.f_trace = None
            return None
        s = frames.get(frame)
        if s is None: return None
        if event == 'line':
            if s['pend'] is not None: s['pend']['locals_changed'] = snap(frame, s['snap'])
            s['pend'] = rec('line', frame)
        elif event == 'return':
            if s['pend'] is not None: s['pend']['locals_changed'] = snap(frame, s['snap'])
            s['pend'] = None
            r = rec('return', frame)
            if frame.f_code.co_name != '<module>': r['locals_changed']['<return>'] = rep(arg)
            st.depth -= 1
            if not frame.f_code.co_flags & 0x20: frames.pop(frame, None)
        elif event == 'exception':
            if s['pend'] is not None: s['pend']['locals_changed'] = snap(frame, s['snap'])
            s['pend'] = None
            rec('exception', frame)['locals_changed']['<exception>'] = clip(f'{arg[0].__name__}: {arg[1]}')
        return local
    def glob(frame, event, arg):
        if event != 'call' or st.stop or frame.f_code.co_filename != fn: return None
        if over(): sys.settrace(None); return None
        st.depth += 1
        top = frame.f_code.co_name == '<module>'
        s = frames.get(frame) or frames.setdefault(frame, dict(snap=modsnap if top else {}, pend=None))
        if top: return local
        rec('call', frame, **snap(frame, s['snap']))
        return local
    prev, err, res, t = sys.gettrace(), None, None, None
    try:
        sys.settrace(glob)
        try:
            exec(body, ns)
            if tail is not None: res = eval(tail, ns)
        finally: sys.settrace(prev)
    except BaseException as e: err, t = e, e.__traceback__
    finally: st.stop = True
    elapsed = time.perf_counter() - st.t0
    frames.clear()
    for k, q in ring.items():
        skips[k]['count'] = max(0, cnt[k] - keep - len(q))
        events.extend(q)
    events.sort(key=lambda e: e['step'])
    trunc = bool(st.reason)
    if len(events) > max_events:
        h = max_events // 2
        events, trunc = events[:h] + events[-h:], True
        st.reason = st.reason or f'kept {max_events} of {len(events) + max_events} events'
    out = dict(events=events, seen=st.seen, truncated=trunc, reason=st.reason, secs=round(elapsed, 4), source=code,
               error=None if err is None else f'{type(err).__name__}: {err}')
    return out, res, err, t

A binary search with the classic bug (`low > high` never stops, `mid` is stuck) is the best test. The trace shows `low`, `high` and `mid` for every pass.

In [ ]:
ns = {}
src = """def bsearch(a, x):
    low, high = 0, len(a) - 1
    while low <= high:
        mid = (low + high) // 2
        if a[mid] < x: low = mid + 1
        elif a[mid] > x: high = mid - 1
        else: return mid
    return -1
r = bsearch([1, 3, 5, 7, 9, 11], 7)
r"""
out, res, err, tb = trace_cell(src, ns)
test_eq(res, 3)
test_eq(err, None)
rows = [e for e in out['events'] if e['event'] == 'line' and e['line'] == 4]
test_eq([e['locals_changed'] for e in rows], [{'mid': '2'}, {'mid': '4'}, {'mid': '3'}])
test_eq([e['locals_changed'] for e in out['events'] if e['line'] == 5], [{'low': '3'}, {}, {}])
test_eq([e['locals_changed'] for e in out['events'] if e['line'] == 6 and e['event'] == 'line'], [{'high': '3'}, {}])

A loop of 10 000 passes keeps the first 50 and the last 50 passes of each line, and a `skip` event with the count between.

In [ ]:
out, res, err, tb = trace_cell("total = 0\nfor i in range(10000):\n    total += i\ntotal", {})
test_eq(res, 49995000)
body = [e for e in out['events'] if e['line'] == 3 and e['event'] == 'line']
test_eq(len(body), 100)
sk = [e for e in out['events'] if e['event'] == 'skip' and e['line'] == 3]
test_eq(sk[0]['count'], 9900)
test_eq(body[-1]['locals_changed'], {'total': '49995000'})

An exception keeps the events up to the error, and the error comes back to the caller.

In [ ]:
out, res, err, tb = trace_cell("x = 1\ny = x / 0\nz = 3", {})
test_eq(type(err), ZeroDivisionError)
test_eq([e['locals_changed'] for e in out['events'] if e['line'] == 1 and e['event'] == 'line'], [{'x': '1'}])
assert 'exception' in [e['event'] for e in out['events']]
assert not [e for e in out['events'] if e['line'] == 3]
assert out['error'].startswith('ZeroDivisionError')

## Running it in a kernel

`run_traced` is a method of every kernel. It sends the tracer as text, runs the cell in the user's namespace, and gets the events back in a `display_data` message with a private mimetype. The person's own output is not changed. If the cell raises, the error is reported as usual.

In [ ]:
#| export
TRACE_MIME = 'application/vnd.kunda.trace+json'

def _harness(code, watch, max_events, max_repr, secs, keep):
    "The cell text that runs `code` under `trace_cell` in the kernel."
    return (inspect.getsource(trace_cell) + f"""
def _kunda_run():
    from IPython.display import display
    out, res, err, tb = trace_cell({code!r}, globals(), {watch!r}, {max_events!r}, {max_repr!r}, {secs!r}, {keep!r})
    display({{{TRACE_MIME!r}: out}}, raw=True)
    if res is not None: __import__('sys').displayhook(res)
    if err is not None:
        while tb is not None and not tb.tb_frame.f_code.co_filename.startswith('<kunda-trace'): tb = tb.tb_next
        raise err.with_traceback(tb)
try: _kunda_run()
finally: del _kunda_run, trace_cell
""")

In [ ]:
#| export
@patch
async def run_traced(self: MessageRouter, code, watch=None, max_events=2000, max_repr=200, secs=5., keep=50, timeout=None, on_output=None, **kw):
    "Run `code` like `execute` and return `AttrDict(ok, error, outputs, execution_count, events, truncated, reason, seen, secs)`. `events` holds one dict per traced line, call, return or exception."
    if getattr(self, 'lang', 'python') != 'python': return AttrDict(ok=False, error='a traced run needs a Python kernel', outputs=[], events=[], truncated=False)
    got = []
    def mine(o): return o.get('output_type') == 'display_data' and TRACE_MIME in o.get('data', {})
    def tell(o):
        if mine(o): got.append(o['data'][TRACE_MIME])
        elif on_output: on_output(o)
    o = await self.execute(_harness(code, watch, max_events, max_repr, secs, keep), on_output=tell, timeout=timeout, **kw)
    got += [x['data'][TRACE_MIME] for x in o.outputs if mine(x)]
    outs = [x for x in o.outputs if not mine(x)]
    t = got[-1] if got else {}
    return AttrDict(ok=o.ok, error=o.error, outputs=outs, execution_count=o.execution_count, lost=o.lost,
                    events=t.get('events', []), truncated=t.get('truncated', False), reason=t.get('reason'),
                    seen=t.get('seen', 0), secs=t.get('secs', 0), source=t.get('source', code))

The real-kernel tests are in `tests/test_trace.py`; they start an ipykernel and run the cells below.